---
title: "Exploratory Data Analysis & Visualization"
execute:
  enabled: true
jupyter: python3
---

## Which hospitals have surprisingly high readmission rates?

You run the data team at the CMS Hospital Readmissions Reduction Program. Your mandate: identify hospitals whose patients are readmitted within 30 days more often than expected, and penalize them. You have readmission data on 3,000+ hospitals across six medical conditions. Where do you start?

:::{.callout-important}
## Definition: Exploratory Data Analysis (EDA)

**Exploratory Data Analysis (EDA)** is the practice of examining a dataset's structure, distributions, and quirks before fitting any model. EDA is the most important step in any analysis — and the one most often skipped.
:::

In Chapter 1, we introduced four modes of reasoning with data: summary, prediction, inference, and causation. This chapter focuses on the first: **summary**. Before you predict, infer, or ask causal questions, you need to understand what's in your data. EDA builds that understanding.

Today we'll explore two real datasets: hospital readmissions and NYC Airbnb listings. Along the way, we'll learn to visualize outliers, missing data patterns, 
and relationships between variables.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

# Load data
DATA_DIR = 'data'

## Part 1: Hospital readmissions

Let's start with the CMS Hospital Readmissions Reduction Program data. Recall from Chapter 1 that this dataset tracks how often patients are readmitted within 30 days, for six medical conditions, across about 3,000 U.S. hospitals.

The EDA checklist:

- **Shape** — how many rows and columns?
- **Types** — what are the column types? Any surprises?
- **Distributions** — what does each variable look like?
- **Missing data** — how much, and why?
- **Relationships** — how do variables relate to each other?

First, we load our data, check how big it is, and check which columns are present.

In [ ]:
hrrp = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hrrp_full.csv')
print(f"Shape: {hrrp.shape}")
print(f"Columns: {list(hrrp.columns)}")

Our analysis always starts here: what does the data look like?

In [ ]:
hrrp.head()

Now let's summarize the numeric columns to get a quick sense of their ranges and typical values.

In [ ]:
# Quick summary of numeric columns
hrrp[['Excess Readmission Ratio', 'Predicted Readmission Rate',
      'Expected Readmission Rate']].describe()

### Shape, types, and first impressions

Let's look a bit deeper: How many rows? How many columns? What types are they? Are there surprises?

In [ ]:
# Data types and non-null counts
hrrp.info()

Scan the `Dtype` column in the output. Most columns are `float64` (numbers), but `Number of Readmissions` is `str`. That's our first clue that something is weird. And `Number of Discharges` has fewer non-null values than other columns. Let's investigate.

### Distributions: what does "normal" look like?

The **Excess Readmission Ratio** is the key metric. It's the ratio of a hospital's predicted readmission rate to its expected rate (ERR = predicted / expected). A value of 1.0 means a hospital's readmission rate matches what's expected given its **patient mix** (the age, health conditions, and severity of the patients it treats). Above 1.0 means more readmissions than expected.

Let's see its distribution.

In [ ]:
# Overall distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(hrrp['Excess Readmission Ratio'].dropna(), bins=50, ax=ax, edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('All Conditions Combined')
plt.tight_layout()
plt.show()

We'll filter the data to show distributions by condition. The `Measure Name` column encodes the medical condition — but it's a bit messy. Let's clean it up for better labels.

In [ ]:
# Distribution by condition
fig, ax = plt.subplots(figsize=(8, 5))
for condition in hrrp['Measure Name'].unique():
    subset = hrrp[hrrp['Measure Name'] == condition]['Excess Readmission Ratio'].dropna()
    label = condition.replace('READM-30-', '').replace('-HRRP', '')
    sns.histplot(subset, bins=30, alpha=0.5, label=label, edgecolor='white', ax=ax)
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('By Condition')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The distributions look roughly symmetric and unimodal, centered near 1.0.
But notice the spread differs by condition — heart failure (HF) has more variation than hip/knee replacement.

:::{.callout-tip}
## Think About It
Why might some conditions have more variability in hospital performance than others?
:::

### Missing data: is it random?

Now EDA gets interesting. How much of the data do you think is missing? 1%? 10%? 50%? Let's count.

In [ ]:
# Missing data summary
missing = hrrp.isnull().sum()
missing_pct = (missing / len(hrrp) * 100).round(1)
pd.DataFrame({'Missing': missing, '% Missing': missing_pct}).query('Missing > 0')

A substantial fraction of `Excess Readmission Ratio` and `Predicted Readmission Rate` values are missing — check the percentages in the table above. Why might they be missing?

Let's check the `Number of Readmissions` column — the one stored as a string.

Let's find the non-numeric entries using `pd.to_numeric()` with `errors='coerce'` — values that can't be converted become NaN.

In [ ]:
# Which values in 'Number of Readmissions' can't be converted to numbers?
non_numeric = hrrp[pd.to_numeric(hrrp['Number of Readmissions'], errors='coerce').isna()
                   & hrrp['Number of Readmissions'].notna()]
non_numeric['Number of Readmissions'].value_counts()

The output reveals that "Too Few to Report" is the culprit. Let's see how these suppressed entries are distributed across conditions. The `.groupby()` method splits the data into groups for aggregation — here, counting suppressed entries per condition.

In [ ]:
# How many "Too Few to Report" entries, by condition?
too_few = hrrp[hrrp['Number of Readmissions'] == 'Too Few to Report']
# :, adds commas to large numbers; :.1f rounds to 1 decimal
print(f"'Too Few to Report' entries: {len(too_few):,} ({len(too_few)/len(hrrp)*100:.1f}%)")
print()
too_few_by_condition = too_few.groupby('Measure Name').size()
total_by_condition = hrrp.groupby('Measure Name').size()
pct_missing = (too_few_by_condition / total_by_condition * 100).round(1)
pd.DataFrame({'Too Few': too_few_by_condition, 'Total': total_by_condition, '% Missing': pct_missing})

A bar chart makes the pattern easier to scan across all six conditions at once. We use pandas `.plot.bar()` here because we're plotting from a pre-computed Series rather than raw data.

In [ ]:
# Visualize the missingness pattern
fig, ax = plt.subplots(figsize=(8, 4))
pct_missing.sort_values().plot.barh(ax=ax, color='coral', edgecolor='white')
ax.set_xlabel('% Suppressed ("Too Few to Report")')
ax.set_title('Missingness varies dramatically by condition')
ax.axvline(x=pct_missing.mean(), color='black', linestyle='--',
           linewidth=1, label=f'Average: {pct_missing.mean():.0f}%')
ax.legend()
plt.tight_layout()
plt.show()

The pattern in the bar chart is a critical finding. Statisticians distinguish three patterns of missingness:

:::{.callout-important}
## Definition: Patterns of Missingness

- **Missing Completely At Random (MCAR)**: no pattern at all. Mold damages a random subset of paper medical records in a basement archive — the destroyed records have nothing in common with each other.
- **Missing At Random (MAR)**: missingness depends on other *observed* data, but not on the missing value itself. Older patients are less likely to complete a follow-up survey, and we have each patient's age on file — so we can account for the gap.
- **Missing Not At Random (MNAR)**: missingness depends on the *unobserved value itself*. Patients with the worst health outcomes are the ones who skip the follow-up survey, and we never observe their outcomes.
:::

Here, CMS suppresses data when counts are too small to report — the data is missing precisely *due to* the small counts. This pattern is **MNAR**, the most dangerous kind: no set of observed variables can fully correct for it.

CABG (coronary artery bypass graft surgery) has ~50% missing — fewer hospitals perform this complex procedure. The hospitals with missing data are systematically different: smaller, often rural, with fewer specialized services.

:::{.callout-important}
## Definition: Bias
A statistical estimate is **biased** when it systematically deviates from the true value. Bias is not random error — it doesn't shrink with more data. Dropping MNAR data introduces **selection bias**: the remaining sample no longer represents the population. An estimate computed from a biased sample will consistently over- or under-estimate the quantity of interest, no matter how large the sample grows.
:::

:::{.callout-warning}
## Dropping missing data can introduce bias
If you drop these rows, you're analyzing only the large, urban hospitals. Your conclusions would be biased — systematically favoring large urban hospitals over the full population.
:::

For now, we're just diagnosing — noticing what's going on. In Chapter 3, we'll decide what to do about it.

:::{.callout-tip}
## Think About It
If you drop all the "Too Few to Report" rows, what kind of hospitals are left in your dataset? What conclusions might change?
:::

### Contingency tables: two categorical variables at once

We've been looking at one variable at a time. A **contingency table** (or cross-tabulation) counts how often each combination of two categorical variables occurs. It's the categorical equivalent of a scatter plot.

Let's see which conditions tend to have suppressed data. We'll create a binary column for whether data was suppressed, then use `pd.crosstab()` to cross-tabulate it with the medical condition.

In [ ]:
# Create a flag for suppressed data
hrrp['suppressed'] = (hrrp['Number of Readmissions'] == 'Too Few to Report')

# Contingency table: condition vs. suppression status
ct = pd.crosstab(hrrp['Measure Name'], hrrp['suppressed'],
                 colnames=['Suppressed'])
ct.index = ct.index.str.replace('READM-30-', '').str.replace('-HRRP', '')
ct

Raw counts are useful, but **proportions** tell a clearer story. What fraction of each condition's records are suppressed?

In [ ]:
# Row proportions: what % of each condition is suppressed?
ct_pct = pd.crosstab(hrrp['Measure Name'], hrrp['suppressed'],
                     colnames=['Suppressed'], normalize='index')
ct_pct.index = ct_pct.index.str.replace('READM-30-', '').str.replace('-HRRP', '')
ct_pct.round(3)

We use `sns.heatmap()` to visualize the contingency table as a color-coded grid:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(ct_pct, annot=True, fmt='.1%', cmap='YlOrRd', ax=ax)
ax.set_title('Suppression rate by condition')
ax.set_ylabel('Condition')
ax.set_xlabel('Suppressed')
plt.tight_layout()
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.show()

Now the pattern pops out: about half of CABG records are suppressed, compared to only ~8% of heart failure records. The contingency table confirms the same finding we saw in the bar chart, but makes the comparison precise and easy to read across all conditions at once.

## Part 2: Airbnb listings

We've seen how missing data can hide systematic bias in healthcare. Now let's see how outliers and skewness can mislead in a completely different domain.

An **outlier** is an observation that falls far from the bulk of the data — either a genuine extreme value (a $999/night penthouse) or a data error (a negative price).

We'll work with 29,000+ Airbnb listings in New York City. An Airbnb host losing bookings to cheaper listings might wonder: how should I reprice? We'll answer that question in Chapter 5, once we have regression tools. For now, we focus on understanding the data itself.

We'll use the Airbnb data to answer a series of increasingly specific questions: What does a typical listing cost? How does price vary across neighborhoods and room types? And if Manhattan listings cost more, does that reflect the borough itself — or the fact that Manhattan has more entire homes?

First, we load our data, check how big it is, and select columns to explore.

In [ ]:
# Load Airbnb data
# low_memory=False prevents mixed-type warnings for large files
airbnb = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False)
print(f"Shape: {airbnb.shape}")

# Select key columns for exploration
cols = ['name', 'neighbourhood_group_cleansed', 'neighbourhood_cleansed',
        'room_type', 'price', 'bedrooms', 'beds', 'security_deposit',
        'number_of_reviews', 'review_scores_rating', 'accommodates']
airbnb = airbnb[cols].copy()
airbnb.head()

### Visualizing missing data

Missing values are data too. Instead of silently dropping them, consider making them visible in your plots.

About 40% of listings have no security deposit recorded. Let's make that missingness visible instead of hiding it.

In [ ]:
# Visualize missingness: security deposit
deposit_status = airbnb['security_deposit'].apply(
    lambda x: f'${x:.0f}' if pd.notna(x) else 'Missing')
fig, ax = plt.subplots(figsize=(8, 4))
deposit_status.value_counts().head(10).plot.bar(ax=ax, edgecolor='white')
ax.set_xlabel('Security Deposit')
ax.set_ylabel('Count')
ax.set_title('Security deposit (40% of listings have no value)')
plt.tight_layout()
plt.show()

Do listings with missing deposits have different prices? Comparing the price distributions reveals whether the missingness is informative.

In [ ]:
airbnb['deposit_missing'] = airbnb['security_deposit'].isna()
fig, ax = plt.subplots(figsize=(8, 4))
for label, group in airbnb[airbnb['price'] > 0].groupby('deposit_missing'):
    tag = 'No deposit listed' if label else 'Deposit listed'
    sns.histplot(group['price'][group['price'] <= 500], bins=40, ax=ax,
                 label=tag, alpha=0.5, stat='density')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Density')
ax.set_title('Price distribution: missing vs. present security deposit')
ax.legend()
plt.tight_layout()
plt.show()

If the distributions differ, the missingness may be informative — a signal, not just a nuisance.

> **Key principle**: Before you handle missing data, you must understand *why* it's missing. The "why" determines the "how."

### The price distribution: beware the right tail

Let's look at the distribution of nightly prices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw distribution
sns.histplot(airbnb['price'], bins=100, ax=axes[0], edgecolor='white')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('All prices')

# Zoomed in
sns.histplot(airbnb[airbnb['price'].between(1, 500)]['price'], bins=50, ax=axes[1], edgecolor='white')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Prices $1–$500 (zoomed in)')

plt.tight_layout()
plt.show()

Look at the left plot: the distribution is so skewed that you can barely see anything. The most expensive listings reach \$999/night, and a long tail of pricey listings dominates the scale.

Now look at the zoomed-in version on the right. Most listings are \$50–\$200/night, with a peak around \$100. Much more informative.

:::{.callout-tip}
## Think About It
Before looking at the numbers below: how large do you think the gap between mean and median price is? $5? $20? $100? What does the size of that gap tell you about the distribution?
:::

In [ ]:
print(f"Mean price:   ${airbnb['price'].mean():.0f}/night")
print(f"Median price: ${airbnb['price'].median():.0f}/night")
print(f"Listings at $0: {(airbnb['price'] == 0).sum()}")
print(f"Max price: ${airbnb['price'].max():,.0f}")
print(f"Listings over $500: {(airbnb['price'] > 500).sum()}")

The mean is noticeably higher than the median — compare the two values printed above. That gap is driven entirely by the right tail: a few hundred expensive listings pull the average up. If you're an Airbnb host trying to price a typical apartment, the mean is misleading. The median is a better summary here.

### Summarizing center and spread

We just saw that the mean and median can disagree. Let's define the key summaries precisely.

:::{.callout-important}
## Definitions: Center and Spread

- **Mean** ($\bar{x}$): the arithmetic average, $\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$. Sensitive to extreme values.
- **Median**: the middle value when observations are sorted. Half the data falls below it, half above. Robust to outliers.
- **Quartiles**: sort the data and split into four equal parts. **Q1** (25th percentile) is the median of the lower half, **Q2** is the median itself (50th percentile), and **Q3** (75th percentile) is the median of the upper half.
- **Interquartile Range (IQR)**: $\text{IQR} = Q3 - Q1$. The width of the middle 50% of the data. Robust to outliers.
:::

Let's see these on the Airbnb price distribution.

In [ ]:
prices = airbnb['price'].dropna()

q1 = prices.quantile(0.25)
median = prices.median()
q3 = prices.quantile(0.75)
mean = prices.mean()

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(prices[prices <= 500], bins=60, ax=ax, color='steelblue', edgecolor='white')

# Vertical lines for mean, median, Q1, Q3
ax.axvline(mean, color='red', linestyle='--', linewidth=2, label=f'Mean = ${mean:.0f}')
ax.axvline(median, color='orange', linestyle='-', linewidth=2, label=f'Median (Q2) = ${median:.0f}')
ax.axvline(q1, color='green', linestyle=':', linewidth=2, label=f'Q1 = ${q1:.0f}')
ax.axvline(q3, color='green', linestyle=':', linewidth=2, label=f'Q3 = ${q3:.0f}')

# Shaded IQR band
ax.axvspan(q1, q3, alpha=0.15, color='green', label=f'IQR = ${q3 - q1:.0f}')

ax.set_xlabel('Price ($)')
ax.set_ylabel('Count')
ax.set_title('Airbnb prices with center and spread markers')
ax.set_xlim(0, 500)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

The mean (red dashed line) is pulled to the right of the median (orange solid line) by the long tail of expensive listings. The shaded green band shows the IQR — the middle 50% of prices. Most listings fall in a fairly narrow range, but the tail extends far to the right.

:::{.callout-important}
## Definition: Robust Statistics

A statistic is **robust** if extreme values barely affect it. The **median** and **interquartile range (IQR)** are robust: moving one observation to \$100,000 hardly changes either one. The **mean** and **standard deviation** are not robust: a single extreme value can shift them dramatically. When a distribution has heavy tails or outliers, robust summaries give a more reliable picture of the typical value and spread.
:::

:::{.callout-warning}
## Silent NA dropping

Functions like `sns.histplot()`, `sns.boxplot()`, and `.describe()` all drop `NaN` values silently — they won't warn you that rows were excluded. Before plotting or summarizing, always check how much data is missing:

```python
df['col'].isna().sum()
```

If a large fraction is missing, your histogram may not represent the full dataset.
:::

> **Key principle**: When a distribution has a heavy tail, the mean doesn't represent a typical value. Always look at the distribution before relying on any single number.

### Log scales for skewed data

The raw distribution is so skewed that standard summaries are misleading. Can we find a better way to see the structure?

When data is heavily right-skewed, the first thing to try is a **log axis**. It shows the same data on a logarithmic scale, compressing the long right tail and stretching out the bunched-up left side — without changing the data itself.

In [ ]:
# Filter out $0 listings
airbnb_pos = airbnb[airbnb['price'] > 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Linear scale — heavily skewed
sns.histplot(airbnb_pos['price'], bins=100, ax=axes[0], edgecolor='white')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Linear scale (raw prices)')

# Panel 2: Log axis with log-spaced bins (equal visual area = equal count)
log_bins = np.logspace(np.log10(airbnb_pos['price'].min()),
                       np.log10(airbnb_pos['price'].max()), 50)
axes[1].hist(airbnb_pos['price'], bins=log_bins, edgecolor='white', color='seagreen')
axes[1].set_xscale('log')
axes[1].set_xlabel('Price ($, log scale)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log axis with log-spaced bins')

# Panel 3: Log-transformed data — for modeling
airbnb_pos['log_price'] = np.log10(airbnb_pos['price'])
sns.histplot(airbnb_pos['log_price'], bins=50, ax=axes[2], edgecolor='white', color='mediumpurple')
axes[2].set_xlabel('log10(Price)')
axes[2].set_ylabel('Count')
axes[2].set_title('Log-transformed data (for modeling)')

plt.tight_layout()
plt.show()

The linear histogram (left) is nearly useless — everything is piled up on the left. The log-axis histogram (center) uses logarithmically-spaced bins so that each bar covers an equal range on the log scale — this spacing ensures bar *areas* are proportional to counts, avoiding the visual distortion that equal-width bins create on a log axis. (We use `axes[1].hist()` from matplotlib here instead of `sns.histplot()`, since custom bin edges require direct access to the matplotlib API.) The log-transformed histogram (right) plots $\log_{10}(\text{price})$ directly — useful when you need to feed the data into a model.

> **Use a log *axis* when you want to *look* at skewed data. Use a log *transform* when you need to *model* it (e.g., in a regression).**

Log scales also make relationships easier to spot. Compare price vs. number of guests on both scales:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sample = airbnb_pos.dropna(subset=['accommodates']).sample(3000, random_state=42)

axes[0].scatter(sample['accommodates'], sample['price'], alpha=0.3, s=10)
axes[0].set_xlabel('Accommodates (guests)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Linear scale: outliers dominate')

axes[1].scatter(sample['accommodates'], sample['price'], alpha=0.3, s=10, color='seagreen')
axes[1].set_yscale('log')
axes[1].set_xlabel('Accommodates (guests)')
axes[1].set_ylabel('Price ($)')
axes[1].set_title('Log scale: trend is clearer')

plt.tight_layout()
plt.show()

On the linear scale, expensive listings in the \$500–\$999 range make it hard to see any pattern among the majority. On the log scale, a clear upward trend emerges: each additional guest corresponds to roughly a multiplicative increase in price. Notice the y-axis still shows dollar amounts — the log axis just spreads them out. We'll use this idea extensively when we build regression models in Chapter 5.

### When scatter plots fail: 2D histograms

With thousands of points, scatter plots become overplotted — every point lands on top of the others and you can't see where the data is concentrated. A **2D histogram** (or heatmap) bins the data in both directions and uses color to show density. We use matplotlib's `.hist2d()` to create one below.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
prices_pos = airbnb[airbnb['price'] > 0]

# Scatter — overplotted
axes[0].scatter(prices_pos['accommodates'], prices_pos['price'],
                alpha=0.1, s=5)
axes[0].set_xlabel('Accommodates (guests)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Scatter plot (overplotted)')
axes[0].set_ylim(0, 700)

# 2D histogram with log color scale to reveal structure in dense regions
from matplotlib.colors import LogNorm
h = axes[1].hist2d(prices_pos['accommodates'][prices_pos['price'] <= 700],
                   prices_pos['price'][prices_pos['price'] <= 700],
                   bins=30, cmap='viridis', norm=LogNorm())
plt.colorbar(h[3], ax=axes[1], label='Count (log scale)')
axes[1].set_xlabel('Accommodates (guests)')
axes[1].set_ylabel('Price ($)')
axes[1].set_title('2D histogram (reveals density)')

plt.tight_layout()
plt.show()

The 2D histogram reveals that the overwhelming majority of listings accommodate 1–4 guests at \$50–\$200/night — a pattern the scatter plot hides behind overplotting.

### Categorical variables: room types and boroughs

Price depends on more than just the listing itself. Which features drive the biggest differences?

Not all data is numeric. Let's look at the categorical variables. So far we've used **histograms** to see the shape of a single numeric variable. Now we need different tools.

The study guide at the end of this chapter includes a reference table for choosing the right plot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Room type
room_counts = airbnb['room_type'].value_counts()
room_counts.plot.bar(ax=axes[0], color=sns.color_palette()[:len(room_counts)], edgecolor='white')
axes[0].set_title('Listings by Room Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Borough
borough_counts = airbnb['neighbourhood_group_cleansed'].value_counts()
borough_counts.plot.bar(ax=axes[1], color=sns.color_palette()[3:8], edgecolor='white')
axes[1].set_title('Listings by Borough')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Why pie charts are almost always a bad choice

Compare the same data in two formats:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
room_counts = airbnb['room_type'].value_counts()

# Pie chart
axes[0].pie(room_counts, labels=room_counts.index, autopct='%1.0f%%',
            startangle=90, colors=sns.color_palette()[:len(room_counts)])
axes[0].set_title('Pie chart')

# Bar chart
room_counts.plot.bar(ax=axes[1], color=sns.color_palette()[:len(room_counts)],
                     edgecolor='white')
axes[1].set_title('Bar chart')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

Which format makes it easier to tell whether "Private room" or "Entire home/apt" is more common? In the pie chart, the two slices look nearly the same size. In the bar chart, the difference is obvious. Pie charts only work well for 3-5 categories where the sizes are very different. For most data, bar charts are clearer.

### Comparing composition across groups

When you want to compare how a categorical variable is distributed across groups, there are three useful views: dodged, stacked, and standardized bar charts.

In [ ]:
ct_room = pd.crosstab(airbnb['neighbourhood_group_cleansed'], airbnb['room_type'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Dodged
ct_room.plot.bar(ax=axes[0], edgecolor='white')
axes[0].set_title('Dodged (raw counts)')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=8)

# Stacked
ct_room.plot.bar(stacked=True, ax=axes[1], edgecolor='white')
axes[1].set_title('Stacked (totals)')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8)

# Standardized (proportions)
ct_room_pct = ct_room.div(ct_room.sum(axis=1), axis=0)
ct_room_pct.plot.bar(stacked=True, ax=axes[2], edgecolor='white')
axes[2].set_title('Standardized (proportions)')
axes[2].set_ylabel('Proportion')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

The dodged chart (left) allows comparisons between each type of listing.
The stacked chart reveals that Manhattan has the most listings overall. The standardized chart (right) shows Manhattan has a higher *proportion* of entire homes than Brooklyn does.

### Faceted plots for aligned comparisons

Now let's look at how variables relate to each other — and whether those relationships are what they seem.

When overlapping histograms get crowded, **faceting** splits the data into separate panels that share the same x-axis. We use `sns.displot()` to create faceted histograms with a shared x-axis:

In [ ]:
prices_filt = airbnb[(airbnb['price'] > 0) & (airbnb['price'] <= 500)]
g = sns.displot(data=prices_filt, x='price', col='room_type',
                col_wrap=1, bins=40, height=3, aspect=1.5)
g.set_titles('{col_name}')
g.set_xlabels('Price ($)')
plt.tight_layout()
plt.show()

Faceting splits the data into separate panels that share the same x or y-axis. 
Comparing distributions - here, prices across room types - is much easier when the shared axis is aligned. 

In [ ]:
# Price by room type and borough
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Price by room type (cap at $500 for visibility)
filtered = airbnb[airbnb['price'].between(1, 500)]
sns.boxplot(data=filtered, x='room_type', y='price', ax=axes[0])
axes[0].set_title('Price by Room Type')
axes[0].set_ylabel('Price ($)')

# Price by borough
sns.boxplot(data=filtered, x='neighbourhood_group_cleansed', y='price', ax=axes[1])
axes[1].set_title('Price by Borough')
axes[1].set_ylabel('Price ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Two patterns jump out:

1. **Entire homes/apartments** cost roughly 2x more than private rooms.
2. **Manhattan** is the most expensive borough. But notice the overlap — there are cheap Manhattan listings and expensive Brooklyn listings.

### What is confounding?

Suppose you survey Stanford undergrads and find that students who attend office hours get higher grades. Should you start going to office hours to boost your GPA?

The association is real — OH attenders genuinely do earn higher grades, on average. But **student motivation** drives both: motivated students are more likely to attend office hours *and* more likely to study hard, do the readings, and start problem sets early. Some of the grade benefit attributed to office hours actually reflects motivation.

Here the **treatment** is attending office hours and the **outcome** is your grade. Motivation is a **confounder**: it affects both the treatment and the outcome, creating an association between them that overstates the causal effect of OH attendance alone.

:::{.callout-important}
## Definition: Confounding
A **confounder** is a variable that affects both the treatment (or exposure) and the outcome, creating an association between them that does not reflect a causal effect of the treatment on the outcome. The association between the treatment and outcome is real — but it doesn't mean the treatment causes the outcome. Confounding is inherently a *causal* concept: it only arises when you interpret an association as evidence for or against a causal claim.
:::

We'll develop formal tools for reasoning about confounders — including directed acyclic graphs (DAGs) and the potential outcomes framework — in Chapters 18–19. For now, the key habit: when you see an association, ask *what else could be driving both variables?*

The same logic applies to the Airbnb data.

:::{.callout-tip}
## Think About It
If someone told you "Manhattan Airbnbs cost more," is that the full story? What if Manhattan has more entire homes and Brooklyn has more private rooms? The comparison might be confounded by room type.
:::

Box plots also clarify the relationship between a numeric and a numeric variable — just discretize one of them. Earlier we saw that accommodates vs. price was hard to read as a scatter plot. Treating accommodates as a categorical variable gives us a box plot that's much clearer:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
box_data = airbnb[(airbnb['price'] > 0) & (airbnb['price'] <= 500) &
                  (airbnb['accommodates'] <= 10)]
sns.boxplot(data=box_data, x='accommodates', y='price', ax=ax)
ax.set_xlabel('Accommodates (guests)')
ax.set_ylabel('Price ($)')
ax.set_title('Price by number of guests')
plt.tight_layout()
plt.show()

The median price roughly doubles from 1-guest to 6-guest listings. The boxes also show that spread increases with group size — larger listings have more price variability.

### Violin plots: seeing the full shape

Box plots summarize distributions with five numbers (min, Q1, median, Q3, max). **Violin plots** show the full distribution shape — revealing patterns that box plots hide. We use `sns.violinplot()` to compare the two side by side:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
filtered = airbnb[airbnb['price'].between(1, 500)]

# Violin plot
sns.violinplot(data=filtered, x='room_type', y='price', ax=axes[1], inner='quartile')
axes[1].set_title('Violin plot')
axes[1].set_ylabel('Price ($)')

# Box plot
sns.boxplot(data=filtered, x='room_type', y='price', ax=axes[0])
axes[0].set_title('Box plot')
axes[0].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

The violin plot reveals that private room prices cluster tightly around \$60, while entire home prices spread broadly from \$100 to \$300. The box plot's median and quartiles can't convey this shape difference.

## Ranking hospitals: a cautionary tale

We noticed that Manhattan is more expensive than Brooklyn — but Manhattan also has more entire homes. Is the borough effect real, or is it confounded by room type? This question mirrors the hospital ranking problem we're about to revisit.

Let's go back to the hospital data for the central finding of this chapter.

Quick reference for the three metrics we'll use:

- **Expected Readmission Rate** = what CMS predicts *should* happen given the patient mix
- **Predicted Readmission Rate** = the hospital's actual estimated rate
- **Excess Readmission Ratio (ERR)** = Predicted / Expected (above 1.0 = worse than expected)

Suppose you want to find the hospitals with the highest readmission rates. A naive approach: just rank by `Predicted Readmission Rate`.

In [ ]:
# Focus on Heart Failure (most common condition, least missing data)
hf = hrrp[hrrp['Measure Name'] == 'READM-30-HF-HRRP'].dropna(
    subset=['Predicted Readmission Rate', 'Expected Readmission Rate', 'Excess Readmission Ratio'])

print(f"Heart failure records: {len(hf):,}")
print()
print(f"Top 10 hospitals by PREDICTED readmission rate:")
top_predicted = hf.nlargest(10, 'Predicted Readmission Rate')[
    ['Facility Name', 'State', 'Predicted Readmission Rate',
     'Expected Readmission Rate', 'Excess Readmission Ratio']]
top_predicted

Now look carefully at the `Expected Readmission Rate` column. These hospitals have *high expected rates* — meaning CMS's model predicts they *should* have high readmissions, given the patients they treat.

The `Excess Readmission Ratio` tells a different story. Let's compare the two rankings.

Here's what to look for in the next plot: the expected rate is on the x-axis, the predicted rate is on the y-axis, and color shows the excess ratio. If predicted = expected, the point falls on the dashed line. Points above the line are doing *worse* than expected; points below are doing *better*.

In [ ]:
# Scatter: Predicted vs Expected rate, colored by Excess Ratio
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(hf['Expected Readmission Rate'],
                     hf['Predicted Readmission Rate'],
                     c=hf['Excess Readmission Ratio'],
                     cmap='coolwarm', alpha=0.5, s=15, vmin=0.85, vmax=1.15)
plt.colorbar(scatter, label='Excess Readmission Ratio')
lims = [hf['Expected Readmission Rate'].min(), hf['Expected Readmission Rate'].max()]
ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1, label='Predicted = Expected')
ax.set_xlabel('Expected Readmission Rate (%)')
ax.set_ylabel('Predicted Readmission Rate (%)')
ax.set_title('Heart Failure: Are "high-rate" hospitals actually underperforming?')
ax.legend()
plt.tight_layout()
plt.show()

The scatter plot reveals a crucial insight. Hospitals with the highest *raw* readmission rates are often treating the sickest patients. Their expected rates are high too. After adjusting for patient mix (via the Excess Readmission Ratio), many of these hospitals are performing *at or below* expectations.

Conversely, some hospitals with modest raw rates are actually doing *worse than expected* given their healthier patient mix.

In [ ]:
# Make it concrete: compare rankings
hf_ranked = hf.copy()
hf_ranked['Rank by Predicted Rate'] = hf_ranked['Predicted Readmission Rate'].rank(ascending=False).astype(int)
hf_ranked['Rank by Excess Ratio'] = hf_ranked['Excess Readmission Ratio'].rank(ascending=False).astype(int)
# Rank change shows how much the naive ranking disagrees with the adjusted ranking
hf_ranked['Rank Change'] = hf_ranked['Rank by Predicted Rate'] - hf_ranked['Rank by Excess Ratio']

# Hospitals that move the most
print("Hospitals that look MUCH WORSE in naive ranking than adjusted ranking:")
print("(Rank change > 0 means the naive ranking is harsher)")
print()
big_movers = hf_ranked.nlargest(5, 'Rank Change')[
    ['Facility Name', 'State', 'Predicted Readmission Rate',
     'Excess Readmission Ratio', 'Rank by Predicted Rate', 'Rank by Excess Ratio']]
big_movers

The naive ranking unfairly penalizes these hospitals. Their high raw rates reflect the complexity of the patients they treat — after adjustment, they're performing fine.

The ranking reversal illustrates **confounding**. Patient severity is a confounder: it affects both which hospital a patient goes to *and* the readmission outcome. The association between hospital identity and high readmission rates is real — those hospitals genuinely have more readmissions. But patient severity drives both the choice of hospital and the readmission outcome, so the association overstates the causal effect of hospital quality. A hospital can rank among the worst on raw rates yet perform at or below expectations after adjusting for its patient mix.

:::{.callout-warning}
## Don't rank before you adjust
The "worst" hospitals might just be the ones treating the hardest cases.
:::

We'll revisit this hospital data in Chapter 18 when we ask a causal question: does the penalty actually reduce readmissions, or does it just punish hospitals that serve vulnerable populations?

## Key Takeaways

- **EDA is the summary mode** — understand your data before modeling it.
- **Always check data types.** A column that looks numeric may be stored as strings. Clean before computing.
- **Distributions reveal what summaries hide.** The mean can be misleading when distributions are skewed. Always plot a histogram. Use a log axis for heavy-tailed data.
- **Missing data tells a story.** Data that's Missing Not At Random (MNAR) can bias your analysis if you drop it. Visualize missingness — don't hide it.
- **An association can be real without being causal.** A confounder drives both the treatment and the outcome, creating an association that doesn't reflect a causal effect. The "worst" hospitals might just be treating the hardest cases.

## Study guide

### Key ideas

- **Exploratory Data Analysis (EDA)**: Examining a dataset's structure, distributions, and quirks before fitting any model.
- **Mean, median, quartiles, IQR**: The mean is the arithmetic average (sensitive to outliers). The median is the middle value (robust). Quartiles divide data into four parts. IQR = Q3 − Q1 measures spread of the middle 50%.
- **Robust statistics**: The median and IQR are robust — barely affected by extreme values. The mean and standard deviation are not.
- **Outlier**: An observation far from the bulk of the data. Can be a genuine extreme or a data error. Outliers can dominate non-robust summaries like the mean.
- **Log axis vs. log transform**: Use a log axis to *visualize* skewed data (keeps original units on the axis). Use a log transform to *model* skewed data (creates a new variable for regression).
- **Patterns of missingness**: **MCAR** — no pattern at all. **MAR** — missingness depends on observed variables. **MNAR** — missingness depends on the unobserved value itself.
- **Confounding**: A confounder is a variable that affects both the treatment and the outcome, creating an association that does not reflect a causal effect. Confounding is a causal concept — it arises when you ask whether X causes Y, not just whether X and Y are correlated.
- Plotting functions (`sns.histplot`, `sns.boxplot`, `.describe()`) drop NAs silently. Always check `df['col'].isna().sum()` first.
- Naive rankings can be unfair: the "worst" hospitals may just be treating the hardest cases.
- Pie charts are almost always a bad choice — use bar charts instead.

### Plot type reference

| Variable types | Plot | When to use |
|----------------|------|-------------|
| One numeric | Histogram | Shape, center, spread |
| One numeric (skewed) | Histogram with log axis | Reveal structure in heavy-tailed data |
| One categorical | Bar chart | Counts/proportions per category |
| One categorical | Pie chart | Almost never — use bar chart instead |
| Two numeric | Scatter plot | Relationships (small-moderate n) |
| Two numeric (dense) | 2D histogram / heatmap | Relationships (large n) |
| Numeric × categorical | Box plot | Compare distributions across groups |
| Numeric × categorical | Faceted histogram / ridge plot | Compare with aligned axes |
| Two categorical | Contingency table | Exact counts/proportions |
| Two categorical | Heatmap of contingency table | Visual comparison |
| Categorical × categorical | Stacked/dodged/standardized bar | Compare composition across groups |

: {.striped .hover}

### Computational tools

- `df.shape`, `df.head()`, `df.info()`, `df.describe()` — first moves in any EDA
- `df['col'].isna().sum()` — count missing values
- `df['col'].value_counts()` — frequency table for categorical data
- `sns.histplot()` — histogram of one numeric variable
- `sns.histplot(x=..., y=..., bins=N)` — 2D histogram / heatmap for two numeric variables
- `sns.boxplot(data=df, x='cat', y='num')` — compare distributions across groups
- `sns.violinplot(data=df, x='cat', y='num')` — compare full distribution shapes across groups
- `sns.countplot()` — bar chart of a categorical variable
- `sns.displot(col='group')` — faceted histograms for comparing distributions
- `pd.crosstab()` — contingency table for two categorical variables
- `sns.heatmap()` — heatmap of a matrix (e.g., contingency table)
- `ax.set_xscale('log')` — set axis to log scale
- `df.plot.bar(stacked=True)` — stacked bar chart

### For the quiz

You should be able to: (1) describe the EDA workflow and why each step matters, (2) explain the difference between MCAR, MAR, and MNAR with an example, (3) explain why the mean is misleading for skewed data, (4) interpret a histogram, box plot, and scatter plot, and (5) explain why adjusting for context matters when comparing groups.